# Wake Word Quickstart — Kaggle / Colab

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/kaggle_quickstart_ww.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/kaggle_quickstart_ww.ipynb)

## Start here

**What it does.** It trains a small wake-word model for any phrase you type (by default "hey mycroft") and then measures it. A text-to-speech model, OmniVoice, speaks the phrase in about two thousand new voices: most are invented from a description (gender, age, accent, pitch), the rest copy the voice of a real person from a speech corpus. Ordinary speech, sound and music from public datasets teach the model what must not wake the assistant. A frozen speech-feature model, WakeHuBERT, turns each 1.5 second clip into features, and a small recurrent network is trained on them. The network is then trained again and again on every stretch of a four-hour stream of speech and sound that it wrongly likes. The detection threshold is chosen on a separate two-hour stream. The model is loaded through the OVOS wake-word plugin and tested on voices it never heard, and on a third, unseen two-hour stream of real speech, sound and music, counting false wake-ups per hour. The run ends by printing what it measured. Those numbers describe this one run; the notebook makes no claim about how the model performs in a real home.

**How long.** About one to one and a half hours on a Kaggle GPU: generating voices takes about twenty minutes; the rest is downloading audio, training, and streaming four hours of audio through the plugin. `SMOKE = True` runs a tiny version in a few minutes on a CPU, with a simple speech synthesizer instead of OmniVoice; it only proves that every step runs.

**What you must set.** Nothing. Internet access must be on (Kaggle: Settings panel, Internet on) and a GPU must be selected (Settings panel, Accelerator, GPU). To train another phrase, type it into `WAKE_WORD`. Any phrase works.

**What you will see at the end.** Detections on held-out synthetic voices, held-out clips that fire by mistake, and false wake-ups per hour with a 95 % interval. The model file `best_f1.onnx` and the OVOS configuration `ovos_config.json` are written to `wakehubert_output`. The threshold is chosen on a held-out stream; the last cell says what that means.

Press **Run All** (Kaggle: Run, then Run All; Colab: Runtime, then Run all).

| Field | Value |
|---|---|
| Inputs | a wake phrase (`WAKE_WORD`) |
| Outputs | `wakehubert_output/best_f1.onnx`, the head in the format `ovos-ww-plugin-wakeforge` loads, and `ovos_config.json` |
| Featurizer | `wakehubert-int8` from `TigreGotico/wakehubert-tiny`, frozen |
| Positives | generated here with OmniVoice (voice design and voice cloning); no published wake-word dataset is used |
| Runtime | GPU about one to one and a half hours; `SMOKE` a few minutes on a CPU |
| Hardware | a GPU (Kaggle T4 or better) for the real run |

Neither `wakeforge` nor its data generator is used; only the plugin, `onnxruntime`, `torch`, `omnivoice` and `huggingface_hub`. CI: linted, compiled under Python 3.11 and 3.12, and executed on the smoke path.

**Read more:**

- [Wake word plugins](https://openvoiceos.github.io/ovos-technical-manual/312-wake_word_plugins/)
- [ovos-ww-plugin-wakeforge](https://github.com/OpenVoiceOS/ovos-ww-plugin-wakeforge)
- [WakeHuBERT featurizer](https://huggingface.co/TigreGotico/wakehubert-tiny)
- [OmniVoice](https://github.com/k2-fsa/OmniVoice)
- [wakeforge](https://github.com/TigreGotico/wakeforge), the full training toolkit

In [ ]:
SMOKE = False      # True: tiny check that every step runs (a few minutes, CPU is enough). False: the real run.
N_SAMPLES = 16     # SMOKE only: how many spoken examples of the phrase to make
EPOCHS = 40        # training epochs; SMOKE trains at least SMOKE_MIN_EPOCHS regardless

## Cell 2 — Settings

**Nothing needs editing: click Run All.** Type another phrase into `WAKE_WORD` to train it.

The recipe:

- **Positives are generated, never downloaded.** OmniVoice speaks the phrase with a new voice for almost every clip: 85 % of the clips use a random description of a voice (gender, age, pitch and an English accent), 15 % copy the voice of a real speaker from LibriSpeech `dev-clean` (40 speakers, about six clips each). The speakers and the descriptions are split before generation: **the test clips use voices the model never trained on**, and a LibriSpeech speaker is never on both sides.
- **Negatives** come from the collection [`TigreGotico/notwakeword-datasets`](https://huggingface.co/collections/TigreGotico/notwakeword-datasets): single spoken words (`not-wake-words-speech-en`) and sounds and music from six of its datasets (`public_domain_sounds_3secs`, `ambient_noises`, `ESC-50`, `FMA_3secs`, `building_106_kitchen_3secs`, `NAR`) plus AudioSet (`agkphysics/AudioSet`, balanced set), read as a few Parquet shards each rather than file by file. Together that is several thousand distinct sound clips. LibriSpeech sentence windows and generated silence are added, and room impulse responses from `davidscripka/MIT_environmental_impulse_responses` give reverb.
- **Hard negatives from a stream.** A separate training stream of about four hours (two of sound and music, two of speech, mostly continuous LibriSpeech sentences) is scanned by the head in 1.5 second windows every 80 ms, exactly as the plugin scans live audio. Every window the head scores above 0.1 becomes a negative and the head is trained again; this repeats until fewer than five windows of the stream score 0.5 or more, at most five times. Each round trains on all the positives, about five times as many negatives, half of them mined and half random.
- **The threshold is chosen on a held-out stream.** A calibration stream of two hours (40 % sound and music), with different speakers, words and sounds from both the training stream and the test stream, is scored by the plugin. The threshold is the lowest of a fixed list (0.5 to 0.999) that gives at most 2 false wake-ups per hour on it, and it is written into the head's metadata and the OVOS configuration. This is not the stream calibration of the wakeforge models.
- **Windows** are 1.5 seconds (24,000 samples, 75 feature frames of 128 numbers), each featurized on its own, as the plugin does.

**What is measured** (all at the chosen threshold, none of it seen in training):

| Measurement | What it is |
|---|---|
| held-out synthetic voices detected | the phrase in voices the model never trained on, out of a stated number of clips |
| held-out clips that fire by mistake | held-out speech words, sound clips and generated silence, out of a stated number |
| false wake-ups per hour on the test stream | the count over two hours of continuous speech, sound and music, with an exact 95 % interval (Poisson), traced to its sources |

The notebook only reports these numbers. A rate of a few wake-ups per hour measured on two hours rests on a handful of events, so the interval is wide.

In [ ]:
import os, time

_STEPS, _last = {}, [time.time()]
def tick(name):
    # records the seconds spent since the previous tick, for the run-time table of Cell 13
    now = time.time()
    _STEPS[name] = now - _last[0]
    _last[0] = now

WAKE_WORD   = os.environ.get("WAKE_WORD", "hey mycroft")
LANGUAGE    = os.environ.get("WW_LANGUAGE", "English")   # language of the phrase, as OmniVoice names it
# Downloaded and generated audio; not part of the result. On Kaggle it goes outside the Output tab.
WORK_DIR    = os.environ.get("WORK_DIR",   "/tmp/wakehubert_work" if os.path.isdir("/kaggle") else "./wakehubert_work")
OUTPUT_DIR  = os.environ.get("OUTPUT_DIR", "./wakehubert_output")    # the model file
SEED        = int(os.environ.get("SEED", "42"))

# Positives (generated with OmniVoice)
N_DESIGN_TRAIN, N_DESIGN_TEST = 1750, 230      # clips with an invented voice
CLONES_PER_SPEAKER = 7                         # clips per real LibriSpeech speaker (30 speakers train, 10 test)
# Negatives
N_SPEECH_TRAIN, N_SPEECH_TEST, N_SPEECH_STREAM = 800, 100, 1200   # single spoken words
N_SENTENCE_WINDOWS = 1200                      # 1.5 s windows cut from LibriSpeech dev-clean sentences
N_SOUND_TEST = 120                              # held-out sound clips for the short synthetic test
N_SILENCE_TRAIN, N_SILENCE_TEST = 300, 40
STREAM_SECONDS = 7200                          # length of each of the calibration and test streams
SOUND_SHARE = 0.4                              # share of a stream that is sound and music rather than speech
MINE_SOUND_SECONDS, MINE_SPEECH_SECONDS = 7200, 7500   # the training stream the head is mined on
LR, BATCH_SIZE = 2e-3, 64
THRESHOLD   = None                             # chosen in Cell 11 on a held-out stream
MAX_MINING_ROUNDS, STOP_BELOW = 5, 5           # mine until fewer than 5 training-stream windows score 0.5 or more, at most 5 rounds
MINE_FROM, MAX_HARD_PER_ROUND, MAX_HARD_TOTAL, MAX_RANDOM_STREAM = 0.1, 6000, 20000, 16000
CANDIDATES = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.97, 0.98, 0.99, 0.995, 0.998, 0.999]

# Threshold rule
CAL_MAX_FA_PER_HOUR = 2.0                      # the threshold is the lowest candidate with at most this many false wake-ups per hour on the calibration stream

SMOKE_MIN_EPOCHS = 30
if SMOKE:
    N_DESIGN_TRAIN, N_DESIGN_TEST = N_SAMPLES, 4
    N_SPEECH_TRAIN, N_SPEECH_TEST, N_SPEECH_STREAM = 2 * N_SAMPLES, 4, 20
    N_SENTENCE_WINDOWS = 0
    N_SOUND_TEST = 6
    N_SILENCE_TRAIN, N_SILENCE_TEST = 2 * N_SAMPLES, 4
    STREAM_SECONDS = MINE_SOUND_SECONDS = MINE_SPEECH_SECONDS = 0
    MAX_MINING_ROUNDS = 1
    EPOCHS = max(EPOCHS, SMOKE_MIN_EPOCHS)
    LR = float(os.environ.get("SMOKE_LR", "5e-3"))
    if EPOCHS == SMOKE_MIN_EPOCHS:
        print(f"SMOKE: training at least {SMOKE_MIN_EPOCHS} epochs")
WORD_SLUG = WAKE_WORD.strip().lower().replace(" ", "_").replace("-", "_")
print(f"Wake word {WAKE_WORD!r} | {EPOCHS} epochs | up to {MAX_MINING_ROUNDS} hard-negative rounds")

## Cell 3 — Install and detect the machine

The cell installs the packages the notebook needs from PyPI: `ovos-ww-plugin-wakeforge` (the plugin that loads the model at the end; it brings `onnxruntime` and `huggingface_hub`), `omnivoice` (the speech generator), `onnx`, `soundfile` and `scipy`. It does **not** install `torch`: Kaggle and Colab ship one that matches their GPU driver, and replacing it breaks the import. Elsewhere, install `torch` yourself first. The `SMOKE` run skips `omnivoice` and needs the `espeak-ng` program instead.

A listed GPU counts only if it runs a calculation. Without a working GPU the real run stops here: generating two thousand voices on a CPU would take hours.

> **Troubleshooting:** a network error means Internet is off (Kaggle: Settings panel, Internet).

In [ ]:
import sys, shutil

_pkgs = ['"ovos-ww-plugin-wakeforge>=0.17.0a1"', '"onnx>=1.16"', '"soundfile>=0.12"', '"scipy>=1.11"', '"huggingface_hub>=0.30"', '"pyarrow>=15"']
if not SMOKE:
    _pkgs.append('"omnivoice>=0.2.1"')
%pip install --quiet --pre {" ".join(_pkgs)}

try:
    import torch
except ImportError as exc:
    raise ImportError("torch is not installed. Kaggle and Colab have it; elsewhere run: pip install torch") from exc

from importlib.metadata import version

_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "colab"      if "google.colab" in sys.modules else
    "paperspace" if os.path.exists("/notebooks")  else
    "local"
)
DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        (torch.ones(8, device="cuda") * 2).sum().item()
        DEVICE = "cuda"
    except Exception as exc:
        print(f"A GPU is listed but cannot run a calculation ({exc}); using the CPU.")
THREADS = min(4, os.cpu_count() or 1)

print(f"Platform : {_platform} | Python {sys.version.split()[0]} | {THREADS} CPU threads")
print(f"torch    : {torch.__version__} | device {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"plugin   : ovos-ww-plugin-wakeforge {version('ovos-ww-plugin-wakeforge')}" + ("" if SMOKE else f" | omnivoice {version('omnivoice')}"))
if not SMOKE:
    assert DEVICE == "cuda", ("The real run needs a GPU to generate voices. Kaggle: Settings panel, Accelerator, GPU. "
                              "Or set SMOKE = True in the second cell for the small check.")
else:
    assert shutil.which("espeak-ng"), "SMOKE speaks the phrase with espeak-ng, which is not installed (Linux: apt-get install espeak-ng)."
os.makedirs(WORK_DIR, exist_ok=True)
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
assert free_gb > (2 if SMOKE else 8), f"Only {free_gb:.1f} GB of disk is free; this run needs about {2 if SMOKE else 8} GB."
print(f"Disk free: {free_gb:.1f} GB")
tick("install and setup")

## Cell 4 — The featurizer

`wakehubert_int8.onnx` is the frozen int8 WakeHuBERT featurizer: 16 kHz audio in, 50 feature frames per second out, each of 128 numbers. It is downloaded at a fixed revision and run with `onnxruntime` on the CPU. The head is trained on its output and the plugin runs the same file, so a head trained on one featurizer is never scored on another.

In [ ]:
import json, math, random, time
from pathlib import Path
import numpy as np
import onnxruntime as ort
import soundfile as sf
from scipy.signal import fftconvolve, resample_poly
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

FEATURIZER_NAME = "wakehubert-int8"
FEATURIZER_REPO = "TigreGotico/wakehubert-tiny"
FEATURIZER_REVISION = "343b26ea8881b5a01447a1659555941b25220d22"
SR, WINDOW = 16000, 24000            # 1.5 s at 16 kHz

feat_path = hf_hub_download(FEATURIZER_REPO, "wakehubert_int8.onnx", revision=FEATURIZER_REVISION)
opts = ort.SessionOptions()
opts.intra_op_num_threads = THREADS
opts.inter_op_num_threads = 1
feat_sess = ort.InferenceSession(feat_path, opts, providers=["CPUExecutionProvider"])
_feat_in = feat_sess.get_inputs()[0].name

def featurize(windows):
    # windows: [n, 24000] float32 -> [n, 75, 128]. Each window is featurized alone, as the plugin does.
    out = []
    for i in range(0, len(windows), 32):
        out.append(feat_sess.run(None, {_feat_in: np.ascontiguousarray(windows[i:i + 32], np.float32)})[0])
    return np.concatenate(out)

probe = featurize(np.zeros((1, WINDOW), np.float32))
N_FRAMES, FEATURE_DIM = probe.shape[1], probe.shape[2]
print(f"Featurizer {FEATURIZER_NAME}: a {WINDOW / SR} s window gives {N_FRAMES} frames of {FEATURE_DIM} numbers")
assert (N_FRAMES, FEATURE_DIM) == (75, 128)

rng = random.Random(SEED)
api = HfApi()

def load16k(path):
    x, sr = sf.read(path, dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if sr != SR:
        g = math.gcd(SR, sr)
        x = resample_poly(x, SR // g, sr // g).astype(np.float32)
    return x

def trim(x, db=-40.0):
    peak = np.abs(x).max()
    if peak == 0:
        return x
    idx = np.nonzero(np.abs(x) > peak * 10 ** (db / 20))[0]
    return x[idx[0]: idx[-1] + 1]

## Cell 5 — Download the negatives, the sentences and the room responses

Only the clips used are downloaded, from the datasets named in Cell 2, in one parallel call per dataset. Held-out sets (never used in training) are set aside for the short test, and the words and sound clips meant for streams are kept apart: one part for the calibration stream, one for the test stream. Sound sets are read from the Parquet shards that Hugging Face keeps for each dataset, and AudioSet from a few of its large shards, so a few thousand clips arrive in a handful of downloads. In the real run, LibriSpeech `dev-clean` (speakers for cloning; sentences for the training negatives and the training stream) and `test-clean` (20 speakers for the calibration stream, 20 other speakers for the test stream) are fetched from openslr.org, about 700 MB.

In [ ]:
import tarfile, urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

AUDIO_EXT = (".wav", ".mp3", ".flac", ".ogg")

def fetch(repo, names, folder):
    names = [n for n in names if not any(c in n for c in "[]*?")]
    snapshot_download(repo, repo_type="dataset", allow_patterns=names, local_dir=Path(WORK_DIR) / folder, max_workers=8)
    return [Path(WORK_DIR) / folder / n for n in names]

def draw_files(repo, n, sub=None):
    files = sorted(f for f in api.list_repo_files(repo, repo_type="dataset")
                   if f.lower().endswith(AUDIO_EXT) and (sub is None or f.startswith(sub)))
    return rng.sample(files, min(n, len(files)))

def load_set(repo, n, folder, sub=None):
    clips = []                                        # [(file name, audio)]
    for f in fetch(repo, draw_files(repo, n, sub), folder):
        try:
            clips.append((f.name, load16k(f)))
        except Exception as exc:                      # an unreadable file is skipped, not fatal
            print(f"skipped {f.name}: {exc}")
    return clips

COLLECTION = "TigreGotico/notwakeword-datasets"
SPEECH_REPO = "TigreGotico/not-wake-words-speech-en"
RIR_REPO = "davidscripka/MIT_environmental_impulse_responses"
# Sound and music sets of the collection, read as a few Parquet shards each (the auto-converted Parquet branch of the dataset),
# not file by file. The number is the most clips taken from a set.
SOUND_SETS = ({"TigreGotico/public_domain_sounds_3secs": 8, "TigreGotico/FMA_3secs": 8, "TigreGotico/NAR": 8, "TigreGotico/ambient_noises": 8} if SMOKE else
              {"TigreGotico/public_domain_sounds_3secs": 2500, "TigreGotico/ambient_noises": 1500, "TigreGotico/ESC-50": 1500,
               "TigreGotico/FMA_3secs": 503, "TigreGotico/building_106_kitchen_3secs": 800, "TigreGotico/NAR": 600})
AUDIOSET = "agkphysics/AudioSet"                       # balanced set; 10 s clips, a few of its 700 MB shards
AUDIOSET_TRAIN_SHARDS, AUDIOSET_TEST_SHARD, AUDIOSET_STREAM_SHARD, AUDIOSET_PER_SHARD = [3, 17, 31], 4, 20, 600

import io
import pyarrow.parquet as pq

def decode(raw):
    x, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if sr != SR:
        g = math.gcd(SR, sr)
        x = resample_poly(x, SR // g, sr // g).astype(np.float32)
    return x

def parquet_clips(repo, filename, cap, tag, revision=None):
    # up to `cap` clips spread evenly over one Parquet shard
    path = hf_hub_download(repo, filename, repo_type="dataset", revision=revision)
    pf = pq.ParquetFile(path)
    col = next(f.name for f in pf.schema_arrow if "bytes" in str(f.type))
    stride, out, k = max(1, pf.metadata.num_rows // cap), [], 0
    for batch in pf.iter_batches(batch_size=64, columns=[col]):
        for item in batch.column(0).to_pylist():
            if k % stride == 0 and len(out) < cap and item.get("bytes"):
                try:
                    x = decode(item["bytes"])
                except Exception:
                    x = None
                if x is not None and len(x) > 0.1 * SR and np.abs(x).max() > 1e-4:
                    out.append((f"{tag}/{Path(item.get('path') or '').name or filename}#{k}", x))
            k += 1
    return out

def split4(clips, r):
    # train / short test / calibration stream / test stream, by clip
    clips = clips[:]
    r.shuffle(clips)
    n = len(clips)
    n_te, n_cal, n_tst = max(1, round(0.04 * n)), max(1, round(0.13 * n)), max(1, round(0.13 * n))
    return clips[n_te + n_cal + n_tst:], clips[:n_te], clips[n_te:n_te + n_cal], clips[n_te + n_cal:n_te + n_cal + n_tst]

def split3(clips, n_train, n_test, n_stream):
    return clips[:n_train], clips[n_train:n_train + n_test], clips[n_train + n_test:n_train + n_test + n_stream]

speech = [(n, trim(x)) for n, x in load_set(SPEECH_REPO, N_SPEECH_TRAIN + N_SPEECH_TEST + N_SPEECH_STREAM, "speech")]
speech = [(n, x) for n, x in speech if len(x) > 0.1 * SR]
sp_train, sp_test, sp_stream = split3(speech, N_SPEECH_TRAIN, N_SPEECH_TEST, N_SPEECH_STREAM)
speech_train, speech_test = [x for _, x in sp_train], [x for _, x in sp_test]
speech_cal, speech_tst = sp_stream[:len(sp_stream) // 2], sp_stream[len(sp_stream) // 2:]   # named; for the calibration and test streams

sounds = {"train": [], "test": [], "cal": [], "tst": []}
t_dl = time.time()
for repo, cap in SOUND_SETS.items():
    short = repo.split("/")[1]
    shards = sorted(f for f in api.list_repo_files(repo, repo_type="dataset", revision="refs/convert/parquet") if f.endswith(".parquet"))
    clips = []
    for fn in shards:
        clips += parquet_clips(repo, fn, -(-cap // len(shards)), short, revision="refs/convert/parquet")
    for key, part in zip(sounds, split4(clips[:cap], random.Random(f"{SEED}:{repo}"))):
        sounds[key] += part
    print(f"  {short}: {len(clips[:cap])} clips ({time.time() - t_dl:.0f} s)")
if not SMOKE:
    files = sorted(f for f in api.list_repo_files(AUDIOSET, repo_type="dataset") if f.endswith(".parquet"))
    bal, ev = [f for f in files if f.startswith("data/bal_train/")], [f for f in files if f.startswith("data/eval/")]
    for i in AUDIOSET_TRAIN_SHARDS:
        sounds["train"] += parquet_clips(AUDIOSET, bal[i], AUDIOSET_PER_SHARD, "AudioSet")
    a = parquet_clips(AUDIOSET, ev[AUDIOSET_TEST_SHARD], AUDIOSET_PER_SHARD, "AudioSet")
    sounds["test"] += a[:N_SOUND_TEST]; sounds["cal"] += a[N_SOUND_TEST:]
    sounds["tst"] += parquet_clips(AUDIOSET, ev[AUDIOSET_STREAM_SHARD], AUDIOSET_PER_SHARD, "AudioSet")
    print(f"  AudioSet: {len(sounds['train'])} clips in all training sounds so far ({time.time() - t_dl:.0f} s)")
noise_train_named, noise_cal, noise_tst = sounds["train"], sounds["cal"], sounds["tst"]
noise_train, noise_test = [x for _, x in sounds["train"]], [x for _, x in sounds["test"]][:max(N_SOUND_TEST, 1)]
rirs = [r[:int(0.6 * SR)] for _, r in load_set(RIR_REPO, 8 if SMOKE else 60, "rir") if len(r) > 100]
print(f"Speech words : {len(speech_train)} train, {len(speech_test)} test, {len(speech_cal)} calibration stream, {len(speech_tst)} test stream")
src_count = lambda lst: ", ".join(f"{k} {v}" for k, v in sorted(Counter(n.split("/")[0] for n, _ in lst).items()))
print(f"Sound clips : {len(noise_train)} train, {len(sounds['test'])} short test, {len(noise_cal)} calibration stream, {len(noise_tst)} test stream | {len(rirs)} room responses")
print("   train: " + src_count(sounds["train"]))
print("   calibration stream: " + src_count(noise_cal))
print("   test stream: " + src_count(noise_tst))

# LibriSpeech
libri = {}
if not SMOKE:
    def get_libri(name):
        tgz = Path(WORK_DIR) / f"{name}.tar.gz"
        if not tgz.exists():
            urllib.request.urlretrieve(f"https://www.openslr.org/resources/12/{name}.tar.gz", tgz)
        with tarfile.open(tgz) as t:
            t.extractall(Path(WORK_DIR) / "librispeech", filter="data")
        tgz.unlink()
        out = {}                                      # speaker -> [(flac path, transcript)]
        for tr in sorted((Path(WORK_DIR) / "librispeech" / "LibriSpeech" / name).glob("*/*/*.trans.txt")):
            spk = tr.parent.parent.name
            for line in tr.read_text().splitlines():
                uid, text = line.split(" ", 1)
                out.setdefault(spk, []).append((tr.parent / f"{uid}.flac", text.lower()))
        return out
    t0 = time.time()
    with ThreadPoolExecutor(2) as ex:
        libri = dict(zip(("dev-clean", "test-clean"), ex.map(get_libri, ("dev-clean", "test-clean"))))
    print(f"LibriSpeech: dev-clean {len(libri['dev-clean'])} speakers, test-clean {len(libri['test-clean'])} speakers ({time.time() - t0:.0f} s)")
    tc = sorted(libri["test-clean"])
    random.Random(SEED).shuffle(tc)
    cal_speakers, tst_speakers = tc[:20], tc[20:]       # the calibration stream and the test stream share no speaker
else:
    cal_speakers = tst_speakers = []
tick("downloads (speech, sounds, LibriSpeech)")

## Cell 6 — Generate the positives

This is where the wake word comes from. For every clip OmniVoice speaks `WAKE_WORD` with a **new voice**:

- **Voice design (85 %).** The voice is described, not copied: gender, age, pitch and an English accent are drawn at random, for example "female, elderly, low pitch, indian accent". The speaking speed is random too, between 1.0 and 1.4 times normal, so that the phrase fits the 1.5 second window; the silence at both ends of a clip is trimmed tightly (35 dB below its peak) before the length check. Each clip is a fresh sample, so no two clips share a voice.
- **Voice cloning (15 %).** The voice of a real person is copied from a few seconds of a LibriSpeech `dev-clean` recording. Each of the 40 speakers gives about six clips. Thirty speakers are for training and ten for the test.

The train and test voices are separate by construction (different clips for designed voices, different speakers for cloned ones). The cell prints the number of distinct voices on each side and the generation speed.

In the `SMOKE` run the `espeak-ng` synthesizer speaks the phrase with random voice variants instead; it is fast but robotic.

Clips still longer than 1.5 seconds are dropped, because the model scores 1.5 second windows; the cell prints how many were kept and dropped. If more than a quarter of the clips are dropped the run stops: the phrase is too long for this window.

In [ ]:
import subprocess, tempfile

ACCENTS = ["american accent", "british accent", "australian accent", "canadian accent", "indian accent"]
POS_TEXT = WAKE_WORD.strip()
pos = {"train": [], "test": []}        # lists of (voice id, 16 kHz audio)
GEN_SECONDS = 0.0

def design_prompt(r):
    parts = [r.choice(["male", "female"]), r.choice(["child", "teenager", "young adult", "middle-aged", "elderly"]),
             r.choice(["very low pitch", "low pitch", "moderate pitch", "high pitch", "very high pitch"])]
    if LANGUAGE.lower() == "english":
        parts.append(r.choice(ACCENTS))
    return ", ".join(parts)

def keep(x, sr=24000):
    g = math.gcd(SR, sr)
    x = trim(resample_poly(np.asarray(x, np.float32), SR // g, sr // g).astype(np.float32), db=-35.0)
    return x if 0.2 * SR <= len(x) <= WINDOW else None

if SMOKE:
    variants = [f"en-{a}+{g}{n}" for a in ("us", "gb") for g in "mf" for n in range(1, 5)]
    rng.shuffle(variants)
    test_variants, train_variants = variants[:4], variants[4:]
    def espeak(voice):
        with tempfile.TemporaryDirectory() as d:
            f = Path(d) / "a.wav"
            subprocess.run(["espeak-ng", "-v", voice, "-p", str(rng.randint(20, 80)), "-s", str(rng.randint(120, 190)),
                            "-w", str(f), POS_TEXT], check=True, capture_output=True)
            x, sr = sf.read(f, dtype="float32")
            return keep(x, sr)
    for split, n, pool in (("train", N_DESIGN_TRAIN, train_variants), ("test", N_DESIGN_TEST, test_variants)):
        for i in range(n):
            v = pool[i % len(pool)]
            x = espeak(v)
            if x is not None:
                pos[split].append((f"espeak:{v}", x))
else:
    from omnivoice import OmniVoice
    t0 = time.time()
    tts = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
    print(f"OmniVoice loaded in {time.time() - t0:.0f} s")
    dropped = 0

    def generate(**kw):
        # A batch with one degenerate item (an empty waveform) is retried item by item; failed items are dropped.
        try:
            return tts.generate(**kw)
        except ValueError:
            n = len(kw["text"])
            out = []
            for i in range(n):
                one = {k: ([v[i]] if isinstance(v, list) else v) for k, v in kw.items()}
                try:
                    out.append(tts.generate(**one)[0])
                except ValueError:
                    out.append(None)
            return out

    def run_design(split, n):
        global dropped, GEN_SECONDS
        r = random.Random(f"{SEED}:{split}:design")
        for start in range(0, n, 32):
            m = min(32, n - start)
            t1 = time.time()
            out = generate(text=[POS_TEXT] * m, language=LANGUAGE, instruct=[design_prompt(r) for _ in range(m)],
                           speed=[round(r.uniform(1.0, 1.4), 2) for _ in range(m)])
            GEN_SECONDS += time.time() - t1
            for j, a in enumerate(out):
                x = None if a is None else keep(a)
                dropped += x is None
                if x is not None:
                    pos[split].append((f"design:{split}:{start + j}", x))

    dev = libri["dev-clean"]
    speakers = sorted(dev)
    random.Random(SEED).shuffle(speakers)
    spk_split = {"train": speakers[:30], "test": speakers[30:40]}

    def run_clone(split):
        global dropped, GEN_SECONDS
        r = random.Random(f"{SEED}:{split}:clone")
        jobs = []
        for spk in spk_split[split]:
            utts = [u for u in dev[spk] if 3.0 <= sf.info(u[0]).duration <= 9.0]
            for k in range(CLONES_PER_SPEAKER):
                jobs.append((spk, r.choice(utts)))
        for start in range(0, len(jobs), 12):
            batch = jobs[start:start + 12]
            refs = [(torch.from_numpy(load16k(u[0])).unsqueeze(0), SR) for _, u in batch]
            t1 = time.time()
            out = generate(text=[POS_TEXT] * len(batch), language=LANGUAGE, ref_audio=refs, ref_text=[u[1] for _, u in batch],
                           speed=[round(r.uniform(1.0, 1.3), 2) for _ in batch])
            GEN_SECONDS += time.time() - t1
            for (spk, _), a in zip(batch, out):
                x = None if a is None else keep(a)
                dropped += x is None
                if x is not None:
                    pos[split].append((f"libri:{spk}", x))

    run_design("train", N_DESIGN_TRAIN); run_design("test", N_DESIGN_TEST)
    run_clone("train"); run_clone("test")
    total = sum(len(v) for v in pos.values()) + dropped
    print(f"Generated {total} clips in {GEN_SECONDS:.0f} s ({total / GEN_SECONDS:.2f} clips/s); dropped {dropped} longer than 1.5 s or too short")
    assert dropped <= 0.25 * total, f"{dropped} of {total} clips were longer than the 1.5 s window; the phrase {POS_TEXT!r} is too long for it."
    del tts
    torch.cuda.empty_cache()

voices = {s: {v for v, _ in pos[s]} for s in pos}
assert not voices["train"] & voices["test"], "a voice is on both sides of the split"
for s in ("train", "test"):
    print(f"{s:5s}: {len(pos[s])} clips, {len(voices[s])} distinct voices")
assert len(pos["train"]) >= (8 if SMOKE else 800) and len(pos["test"]) >= 4
tick("generate positives")

## Cell 7 — Make the 1.5-second windows

Every example becomes one window of exactly 24,000 samples:

- a spoken clip (shorter than the window) is placed at a random position in silence;
- a longer clip is cropped at a random position;
- training windows get reverb with 30 % probability (a recorded room response), a random gain and, with 80 % probability, background sound from the training sound clips at 0 to 25 dB below the signal. This applies to the wake phrase **and** to the speech negatives, so the head learns both that the phrase in noise is still the phrase and that speech in noise is still not;
- each training clip is used several times at different positions and levels; test windows are used once, clean, at a fixed seed.

The silence examples are windows of digital zeros and of white noise at -60 and -40 dB relative to full scale. Without them a head can fire on the first, empty buffer of a quiet microphone. The sentence windows are random 1.5 second pieces of LibriSpeech sentences, often cut through the middle of a word.

In [ ]:
def peak_to(x, level):
    p = np.abs(x).max()
    return x * (level / p) if p > 0 else x

def fit_window(x, r):
    if len(x) >= WINDOW:
        s = int(r.integers(0, len(x) - WINDOW + 1))
        return x[s:s + WINDOW].copy()
    w = np.zeros(WINDOW, np.float32)
    s = int(r.integers(0, WINDOW - len(x) + 1))
    w[s:s + len(x)] = x
    return w

def augment(w, r, noises):
    if rirs and r.random() < 0.3:                      # reverb: convolve with a recorded room response
        peak = np.abs(w).max()
        w = fftconvolve(w, rirs[int(r.integers(len(rirs)))])[:WINDOW]
        w = w * (peak / (np.abs(w).max() + 1e-8))
    w = w * 10 ** (r.uniform(-9, 3) / 20)
    if noises and r.random() < 0.8:
        bg = fit_window(noises[int(r.integers(len(noises)))], r)
        bg_rms, w_rms = np.sqrt((bg ** 2).mean()) + 1e-8, np.sqrt((w ** 2).mean()) + 1e-8
        w = w + bg * (w_rms / bg_rms) * 10 ** (-r.uniform(0, 25) / 20)
    return np.clip(w, -1, 1).astype(np.float32)

def silence_windows(n, r):
    kinds = [None, -60.0, -40.0]
    out = []
    for i in range(n):
        level = kinds[i % 3]
        out.append(np.zeros(WINDOW, np.float32) if level is None
                   else np.clip(r.standard_normal(WINDOW) * 10 ** (level / 20), -1, 1).astype(np.float32))
    return out

# Sentence windows from LibriSpeech dev-clean (continuous speech, cut at random)
sentence_clips = []
if N_SENTENCE_WINDOWS:
    utts = [u for spk in libri["dev-clean"].values() for u in spk]
    for path, _ in random.Random(SEED).sample(utts, min(N_SENTENCE_WINDOWS, len(utts))):
        sentence_clips.append(load16k(path))

def training_set(r, extra=(), variants=(3, 2, "auto", 1)):
    pools = [(1, [peak_to(x, 0.7) for _, x in pos["train"]], variants[0]),
             (0, [peak_to(x, 0.7) for x in speech_train], variants[1]),
             (0, [peak_to(x, float(r.uniform(0.02, 0.8))) for x in noise_train], variants[2]),
             (0, [peak_to(x, float(r.uniform(0.1, 0.8))) for x in sentence_clips], variants[3])]
    wins, labels, groups = [], [], []
    for label, clips, times in pools:
        for gi, clip in enumerate(clips):
            for _ in range((2 if len(clip) > 5 * SR else 1) if times == "auto" else times):   # long clips give two crops
                wins.append(augment(fit_window(clip, r), r, noise_train))
                labels.append(label)
                groups.append(gi % 10)
    for w in silence_windows(N_SILENCE_TRAIN, r):
        wins.append(w); labels.append(0); groups.append(int(r.integers(10)))
    for w in extra:                                    # mined hard negatives
        for _ in range(5):
            wins.append(augment(w, r, noise_train)); labels.append(0); groups.append(int(r.integers(10)))
    return np.stack(wins), np.array(labels, np.float32), np.array(groups)

def test_set(r):
    sets = {
        "wake phrase": [fit_window(peak_to(x, 0.7), r) for _, x in pos["test"]],
        "speech": [fit_window(peak_to(x, 0.7), r) for x in speech_test],
        "noise": [fit_window(peak_to(x, float(r.uniform(0.02, 0.8))), r) for x in noise_test],
        "silence": silence_windows(N_SILENCE_TEST, r),
    }
    return {k: np.stack(v) for k, v in sets.items()}

train_audio, train_labels, train_groups = training_set(np.random.default_rng(SEED))
test_audio = test_set(np.random.default_rng(SEED + 1))
print(f"Training windows: {len(train_audio)} ({int(train_labels.sum())} wake phrase, {int((1 - train_labels).sum())} other)")
print("Test windows    : " + ", ".join(f"{len(v)} {k}" for k, v in test_audio.items()))
t0 = time.time()
train_feats = featurize(train_audio)
print(f"Featurized {len(train_feats)} training windows in {time.time() - t0:.0f} s -> {train_feats.shape}")
tick("windows and features")

## Cell 8 — Three streams

Three continuous streams are built from real speech and sound. Each segment (a LibriSpeech sentence, a spoken word, a sound clip) gets a random level and is followed by 0.3 seconds of silence. Each stream records where every segment starts and what it is, so that a false wake-up can be traced to its source.

| Stream | Used for | Sources |
|---|---|---|
| training stream (about 4 h) | mining hard negatives | LibriSpeech `dev-clean` sentences, the training speech words, two hours of the training sounds and music |
| calibration stream (2 h, 40 % sound and music) | choosing the threshold | 20 `test-clean` speakers, half of the stream words, held-out sounds and music |
| test stream (2 h, 40 % sound and music) | the reported false wake-ups per hour | 20 other `test-clean` speakers, the other half of the words, other held-out sounds and music |

The three streams share no clip, word or speaker. None of them contains the wake phrase.

In [ ]:
from collections import Counter

def sentence_parts(speakers, split, seconds, seed):
    utts = [(spk, u) for spk in speakers for u in libri[split][spk]]
    random.Random(seed).shuffle(utts)
    parts, total = [], 0.0
    for spk, (path, text) in utts:
        if total >= seconds:
            break
        x = load16k(path)
        total += len(x) / SR
        parts.append(("LibriSpeech sentence", f"{spk}/{path.stem}: {text}", x))
    return parts

word_parts = lambda named: [("speech word", n, x) for n, x in named]
sound_parts = lambda named: [("sound: " + n.split("/")[0], n, x) for n, x in named]
duration = lambda parts: sum(len(x) for *_, x in parts) / SR

def build_stream(parts, seed):
    r = np.random.default_rng(seed)
    gap = np.zeros(int(0.3 * SR), np.float32)
    audio, index, t = [], [], 0.0
    for i in r.permutation(len(parts)):
        src, name, x = parts[i]
        x = peak_to(x, float(r.uniform(0.05, 0.8) if src.startswith("sound") else r.uniform(0.2, 0.8)))
        index.append((t, t + len(x) / SR, src, name))
        audio += [x, gap]
        t += len(x) / SR + 0.3
    return np.concatenate(audio).astype(np.float32), index

def pick_sounds(named, seconds, seed):
    # a random subset of sound clips adding up to `seconds` (all of them when seconds is 0)
    named = named[:]
    random.Random(seed).shuffle(named)
    if not seconds:
        return named
    out, total = [], 0.0
    for n, x in named:
        if total >= seconds:
            break
        out.append((n, x)); total += len(x) / SR
    return out

train_parts = word_parts(sp_train) + sound_parts(pick_sounds(noise_train_named, MINE_SOUND_SECONDS, SEED + 10))
if MINE_SPEECH_SECONDS:
    train_parts += sentence_parts(sorted(libri["dev-clean"]), "dev-clean", MINE_SPEECH_SECONDS - duration(word_parts(sp_train)), SEED + 11)
mine_audio, mine_index = build_stream(train_parts, SEED + 12)

cal_parts = word_parts(speech_cal) + sound_parts(pick_sounds(noise_cal, SOUND_SHARE * STREAM_SECONDS, SEED + 17))
tst_parts = word_parts(speech_tst) + sound_parts(pick_sounds(noise_tst, SOUND_SHARE * STREAM_SECONDS, SEED + 18))
if STREAM_SECONDS:
    cal_parts += sentence_parts(cal_speakers, "test-clean", STREAM_SECONDS - duration(cal_parts), SEED + 13)
    tst_parts += sentence_parts(tst_speakers, "test-clean", STREAM_SECONDS - duration(tst_parts), SEED + 14)
cal_audio, cal_index = build_stream(cal_parts, SEED + 15)
tst_audio, tst_index = build_stream(tst_parts, SEED + 16)
CAL_HOURS, STREAM_HOURS = len(cal_audio) / SR / 3600, len(tst_audio) / SR / 3600
for name, a, ix in (("training", mine_audio, mine_index), ("calibration", cal_audio, cal_index), ("test", tst_audio, tst_index)):
    kinds = Counter()
    for t0_, t1_, src, _ in ix:
        kinds[src.split(":")[0]] += t1_ - t0_
    nonspeech = sum(v for k, v in kinds.items() if k == "sound")
    print(f"{name:11s} stream: {len(a) / SR / 3600:.2f} h, {len(ix)} segments, {100 * nonspeech / max(1, sum(kinds.values())):.0f} % sound and music (" + ", ".join(f"{k} {v / 60:.0f} min" for k, v in kinds.items()) + ")")
tick("streams")

## Cell 9 — Train the head, then mine hard negatives from the stream

The head is a single GRU layer with 128 hidden units followed by one linear layer; its last output is the score of the window (a logit, turned into a probability by a sigmoid). It is plain `torch`, trained with the Adam optimiser. A tenth of the windows are kept apart for validation, and the epoch with the lowest validation loss is kept.

**Round 0** trains on the windows of Cell 7. Then, each round:

1. The head scores the whole training stream in 1.5 second windows every 80 ms, which is how the plugin scores live audio. The features of the stream are computed once and kept.
2. Every window scoring above 0.1 is a candidate hard negative. The highest scoring ones are taken, at most 6,000 per round and never two within 0.24 seconds of each other, since neighbouring windows are nearly the same sound.
3. The head is trained again from scratch on all positives plus about five times as many negatives: the mined windows of every round so far (up to 20,000) and an equal number of random windows of the same stream, in addition to the base negatives of Cell 7.

The rounds stop when fewer than five windows of the training stream score 0.5 or more (at most five rounds). The test stream and the calibration stream are never scored here.

In [ ]:
import torch.nn as nn

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(FEATURE_DIM, 128, batch_first=True)
        self.fc = nn.Linear(128, 1)
    def forward(self, features):                 # [batch, frames, 128] -> [batch, 1] logit
        out, _ = self.gru(features)
        return self.fc(out[:, -1])

def train_head(feats, labels, groups, tag):
    torch.manual_seed(SEED)
    head = Head().to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()
    X = torch.from_numpy(feats).to(DEVICE)
    Y = torch.from_numpy(labels).to(DEVICE).unsqueeze(1)
    is_val = torch.from_numpy(groups == 0).to(DEVICE)
    if is_val.sum() == 0 or is_val.all():
        is_val = torch.arange(len(X), device=DEVICE) % 10 == 0
    Xtr, Ytr, Xva, Yva = X[~is_val], Y[~is_val], X[is_val], Y[is_val]
    best = (float("inf"), None)
    for epoch in range(1, EPOCHS + 1):
        head.train()
        perm = torch.randperm(len(Xtr), device=DEVICE)
        for i in range(0, len(perm), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            opt.zero_grad()
            loss_fn(head(Xtr[idx]), Ytr[idx]).backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            val_logits = head(Xva)
            val_loss = loss_fn(val_logits, Yva).item()
            val_acc = ((val_logits > 0).float() == Yva).float().mean().item()
        if val_loss < best[0]:
            best = (val_loss, {k: v.detach().cpu().clone() for k, v in head.state_dict().items()})
        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
            print(f"[{tag}] epoch {epoch:3d}/{EPOCHS}  validation loss {val_loss:.4f}  accuracy {val_acc:.3f}")
    head.load_state_dict(best[1])
    del X, Y, Xtr, Xva
    return head, best

def score_feats(head, feats16, bs=2048):
    head.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(feats16), bs):
            x = torch.from_numpy(feats16[i:i + bs].astype(np.float32)).to(DEVICE)
            out.append(torch.sigmoid(head(x)).squeeze(1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0, np.float32)

def score_windows(head, wins):
    return score_feats(head, featurize(wins))

HOP = 1280                                             # 80 ms
def stream_features(audio):
    starts = np.arange(0, len(audio) - WINDOW + 1, HOP)
    feats = np.empty((len(starts), N_FRAMES, FEATURE_DIM), np.float16)
    for i in range(0, len(starts), 64):
        b = np.stack([audio[s:s + WINDOW] for s in starts[i:i + 64]])
        feats[i:i + len(b)] = featurize(b)
    return feats

def pick_hard(scores, floor, limit, min_gap=3):
    chosen, taken = [], np.zeros(len(scores), bool)
    for i in np.argsort(-scores):
        if scores[i] < floor or len(chosen) >= limit:
            break
        if taken[max(0, i - min_gap): i + min_gap + 1].any():
            continue
        taken[i] = True
        chosen.append(int(i))
    return chosen

t0 = time.time()
head, best = train_head(train_feats, train_labels, train_groups, "round 0")
mine_feats = stream_features(mine_audio)
mine_hours = len(mine_audio) / SR / 3600
print(f"Featurized the training stream: {len(mine_feats)} windows ({mine_hours:.2f} h) in {time.time() - t0:.0f} s")
pos_f, base_neg = train_feats[train_labels == 1], train_feats[train_labels == 0]
mr = np.random.default_rng(SEED + 21)
rand_idx = mr.choice(len(mine_feats), size=min(MAX_RANDOM_STREAM, len(mine_feats)), replace=False)
hard_idx = set()
for rnd in range(1, MAX_MINING_ROUNDS + 1):
    scores = score_feats(head, mine_feats)
    n_half = int((scores >= 0.5).sum())
    print(f"[round {rnd}] training stream: {n_half} of {len(scores)} windows score at least 0.5, {int((scores >= MINE_FROM).sum())} at least {MINE_FROM}")
    if rnd > 1 and n_half < STOP_BELOW:
        print(f"Fewer than {STOP_BELOW} windows score 0.5 or more: mining stops.")
        break
    chosen = pick_hard(scores, MINE_FROM, MAX_HARD_PER_ROUND)
    hard_idx |= set(chosen)
    if len(hard_idx) > MAX_HARD_TOTAL:
        hard_idx = set(mr.choice(sorted(hard_idx), size=MAX_HARD_TOTAL, replace=False).tolist())
    print(f"[round {rnd}] mined {len(chosen)} windows; {len(hard_idx)} hard negatives in total")
    hard = sorted(hard_idx)
    feats = np.concatenate([pos_f, base_neg, mine_feats[rand_idx].astype(np.float32), mine_feats[hard].astype(np.float32)])
    labels = np.concatenate([np.ones(len(pos_f)), np.zeros(len(feats) - len(pos_f))]).astype(np.float32)
    groups = mr.integers(0, 10, len(feats))
    head, best = train_head(feats, labels, groups, f"round {rnd}")
    del feats
scores = score_feats(head, mine_feats)
print(f"[final] training stream: {int((scores >= 0.5).sum())} of {len(scores)} windows still score at least 0.5")
TRAIN_SECONDS = time.time() - t0
print(f"Training and mining took {TRAIN_SECONDS:.0f} s on {DEVICE}; kept the epoch with validation loss {best[0]:.4f}")
tick("train and mine")

## Cell 10 — Export in the plugin's format

`ovos-ww-plugin-wakeforge` loads a head from a file path. It expects an ONNX graph that maps `[batch, frames, 128]` features to one logit per window, and it reads these keys from the file's metadata:

| Metadata key | Value here | Meaning |
|---|---|---|
| `pretrained_featurizer` | `wakehubert-int8` | the featurizer the plugin downloads and runs before the head |
| `featurizer_revision` | the revision used above | the exact featurizer files |
| `window_frames` | `75` | frames scored per window (1.5 s) |
| `default_threshold` | set in Cell 11 | the threshold chosen on the held-out stream |

The head is exported twice: first without a threshold, so that Cell 11 can record the raw scores the plugin gives on the calibration stream, and again with the chosen threshold. A head that states a `default_threshold` is scored by the plugin the way the ready-made models are: it fires on a single 80 ms score at or above the threshold, with at least two seconds between wake-ups. The cell checks that `onnxruntime` and `torch` give the same logits.

In [ ]:
import onnx

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
HEAD_PATH = Path(OUTPUT_DIR) / "best_f1.onnx"
head_cpu = Head()
head_cpu.load_state_dict({k: v.cpu() for k, v in head.state_dict().items()})
head_cpu.eval()

def export_head(threshold=None):
    example = torch.from_numpy(train_feats[:2])
    try:
        torch.onnx.export(head_cpu, example, str(HEAD_PATH), input_names=["features"], output_names=["logits"],
                          dynamic_axes={"features": {0: "batch", 1: "frames"}, "logits": {0: "batch"}},
                          opset_version=17, dynamo=False)
    except TypeError:                                  # older torch has no dynamo argument
        torch.onnx.export(head_cpu, example, str(HEAD_PATH), input_names=["features"], output_names=["logits"],
                          dynamic_axes={"features": {0: "batch", 1: "frames"}, "logits": {0: "batch"}}, opset_version=17)
    model = onnx.load(str(HEAD_PATH))
    meta = {
        "pretrained_featurizer": FEATURIZER_NAME, "featurizer_revision": FEATURIZER_REVISION,
        "window_frames": N_FRAMES, "feature_dim": FEATURE_DIM, "frame_rate_hz": 50.0, "arch": "gru",
        "wake_word": WAKE_WORD, "license": "Apache-2.0",
        "training_data": f"positives generated with OmniVoice ({'espeak-ng in SMOKE' if SMOKE else 'voice design and cloning of LibriSpeech dev-clean speakers'}); negatives from {COLLECTION} and LibriSpeech dev-clean; room responses from {RIR_REPO}; generated silence",
    }
    if threshold is not None:
        meta["default_threshold"] = repr(float(threshold))
        meta["threshold_note"] = f"chosen on a held-out stream of {CAL_HOURS:.2f} h for at most {CAL_MAX_FA_PER_HOUR} false wake-ups per hour; not stream-calibrated"
    for key, value in meta.items():
        entry = model.metadata_props.add()
        entry.key, entry.value = key, str(value)
    onnx.save(model, str(HEAD_PATH))

export_head()
check = ort.InferenceSession(str(HEAD_PATH), opts, providers=["CPUExecutionProvider"])
for n in (75, 40):                                  # the plugin may score other frame counts
    f = train_feats[:8, :n]
    got = check.run(None, {"features": f})[0].reshape(-1)
    with torch.no_grad():
        want = head_cpu(torch.from_numpy(f)).numpy().reshape(-1)
    assert np.abs(got - want).max() < 1e-3, f"onnxruntime and torch disagree at {n} frames"
print(f"Wrote {HEAD_PATH} ({HEAD_PATH.stat().st_size / 1024:.0f} KB); onnxruntime matches torch")

## Cell 11 — Choose the threshold on the held-out calibration stream

The plugin scores the calibration stream block by block, in one unbroken feed, and the cell keeps the raw score it gives to every 80 ms block. A wake-up is then a block at or above a threshold, followed by two seconds in which the plugin ignores further scores; the cell replays that rule for each of the candidate thresholds 0.5 to 0.999 (the same scores, so the replay is exact; the real plugin runs on the test stream in Cell 13). **The threshold is the lowest candidate with at most 2 false wake-ups per hour** on this stream. The calibration stream shares no speaker, word or sound with the training stream or the test stream, and it holds no wake phrase, so every wake-up on it is false. Beside each candidate the table prints the recall on the held-out synthetic voices (they are not used to pick the threshold), so that the trade-off between false wake-ups and recall is visible.

Before the choice, the cell traces the wake-ups at threshold 0.5 back to their sources (sentences, words, sounds): that shows where the negatives are thin.

This is a threshold chosen on a held-out stream of a stated length. It is **not** the stream calibration of the published models, which needs a much longer stream and a newer wakeforge.

In [ ]:
from ovos_ww_plugin_wakeforge import WakeForgeHotwordPlugin

OVOS_BASE = {"module": "ovos-ww-plugin-wakeforge", "model": str(HEAD_PATH), "onnx_threads": THREADS}

def to_int16(a):
    return (np.clip(a, -1, 1) * 32767).astype(np.int16)

def stream_scores(audio, label):
    # the plugin's raw score for every 80 ms block of an unbroken feed (a threshold of 2 never fires)
    pcm = to_int16(audio)
    pl = WakeForgeHotwordPlugin(key_phrase=WORD_SLUG, config={**OVOS_BASE, "threshold": 2.0})
    scores, t1 = [], time.time()
    for k, i in enumerate(range(0, len(pcm) - HOP + 1, HOP)):
        pl.update(pcm[i:i + HOP].tobytes())
        scores.append(pl.last_score or 0.0)
        if k and k % 9000 == 0:
            print(f"  {label}: scored {i / SR / 60:.0f} min ({time.time() - t1:.0f} s)")
    return np.array(scores)

def simulate(scores, threshold):
    # what a head that states default_threshold does: fire on one score >= threshold, then ignore 2 s
    fires, quiet = [], 0
    for k, sc in enumerate(scores):
        if quiet > 0:
            quiet -= HOP
            continue
        if sc >= threshold:
            fires.append((k + 1) * HOP / SR)
            quiet = 2 * SR
    return fires

from scipy.stats import chi2

def poisson_ci(k, hours):
    # exact 95 % interval for a rate, from k events in `hours` hours
    lo = 0.0 if k == 0 else chi2.ppf(0.025, 2 * k) / 2 / hours
    hi = chi2.ppf(0.975, 2 * k + 2) / 2 / hours
    return lo, hi

def by_source(fires, index, title, top=8):
    starts = np.array([t0_ for t0_, *_ in index])
    seconds = Counter()
    for t0_, t1_, src, _ in index:
        seconds[src] += t1_ - t0_
    count, offenders = Counter(), Counter()
    for t_ in fires:
        k = max(0, int(np.searchsorted(starts, t_, side="right")) - 1)
        count[index[k][2]] += 1
        offenders[(index[k][2], index[k][3])] += 1
    print(f"{title}: {len(fires)} wake-ups")
    for src in sorted(seconds):
        print(f"   {src:<42s} {count[src]:>4d} wake-ups in {seconds[src] / 3600:.2f} h")
    for (src, name), n in offenders.most_common(top):
        print(f"   {n}x  {name[:110]}")

t0 = time.time()
cal_scores = stream_scores(cal_audio, "calibration stream")
print(f"Scored the {CAL_HOURS:.2f} h calibration stream in {time.time() - t0:.0f} s")
by_source(simulate(cal_scores, 0.5), cal_index, "\nCalibration stream at threshold 0.5")

def clip_peak(pl, window):
    pl.reset()
    pcm, peak = to_int16(np.concatenate([window, np.zeros(int(0.4 * SR), np.float32)])), 0.0
    for i in range(0, len(pcm) - HOP + 1, HOP):
        pl.update(pcm[i:i + HOP].tobytes())
        peak = max(peak, pl.last_score or 0.0)
    return peak

# the highest score the plugin gives each held-out synthetic wake-phrase clip; a clip is detected at a threshold when it reaches it
pl_peak = WakeForgeHotwordPlugin(key_phrase=WORD_SLUG, config={**OVOS_BASE, "threshold": 2.0})
pos_peaks = np.array([clip_peak(pl_peak, w) for w in test_audio["wake phrase"]])
print(f"\n{'threshold':>10} {'wake-ups':>9} {'per hour':>9}   {'recall on held-out synthetic voices':>36}")
THRESHOLD = None
for cand in CANDIDATES:
    n = len(simulate(cal_scores, cand))
    print(f"{cand:>10} {n:>9} {n / CAL_HOURS:>9.2f}   {int((pos_peaks >= cand).sum()):>5} of {len(pos_peaks)} = {(pos_peaks >= cand).mean():>6.1%}")
    if THRESHOLD is None and n / CAL_HOURS <= CAL_MAX_FA_PER_HOUR:
        THRESHOLD = cand
CAL_REACHED = THRESHOLD is not None
if not CAL_REACHED:
    THRESHOLD = CANDIDATES[-1]
    print(f"No candidate reaches {CAL_MAX_FA_PER_HOUR} per hour on the calibration stream; using the highest, {THRESHOLD}.")
CAL_FA = len(simulate(cal_scores, THRESHOLD)) / CAL_HOURS
cal_lo, cal_hi = poisson_ci(len(simulate(cal_scores, THRESHOLD)), CAL_HOURS)
print(f"\nThreshold chosen on a held-out stream ({CAL_HOURS:.2f} h): {THRESHOLD} ({CAL_FA:.2f} false wake-ups per hour on it, 95 % interval {cal_lo:.2f} to {cal_hi:.2f})")
export_head(THRESHOLD)
OVOS_HOTWORD = {**OVOS_BASE, "threshold": THRESHOLD}
# The OVOS configuration is written here, so that the output holds it before the measurements.
cfg = {
    "listener": {"wake_word": WORD_SLUG},
    "hotwords": {WORD_SLUG: {
        "module": "ovos-ww-plugin-wakeforge",
        "model": "~/.local/share/mycroft/best_f1.onnx",     # where you copied the file to
        "threshold": THRESHOLD,                              # chosen on a held-out stream; also stored in the file
    }},
}
(Path(OUTPUT_DIR) / "ovos_config.json").write_text(json.dumps(cfg, indent=2) + "\n")
print(f"Wrote {Path(OUTPUT_DIR) / 'ovos_config.json'} and {HEAD_PATH}")
tick("threshold (calibration stream)")

## Cell 12 — Test through the OVOS plugin

`WakeForgeHotwordPlugin` is the class OVOS creates from the `hotwords` section of `mycroft.conf`. Every test below builds it from the configuration printed in Cell 14 (the model by file path, the chosen threshold set explicitly, which is also stated in the head's metadata) and feeds it audio the way the listener does: 16-bit samples in 80 ms blocks. Two tests:

1. **Synthetic held-out voices.** Short windows of the phrase in voices never trained on, and of held-out speech, sounds and silence. A window counts as detected if `found_wake_word()` returns `True` at any point.
2. **The test stream, with no reset.** Two hours of real speech, sound and music in one unbroken feed, which no earlier step has seen: 20 `test-clean` speakers who are not in the calibration stream, held-out speech words and held-out sounds and music. Every wake-up is a false one, because the phrase is not in the stream. The result is false wake-ups per hour, traced back to its sources.

In [ ]:
from ovos_ww_plugin_wakeforge import WakeForgeHotwordPlugin

def to_pcm(window, tail=0.4):
    pcm = (np.clip(window, -1, 1) * 32767).astype(np.int16)
    return np.concatenate([pcm, np.zeros(int(tail * SR), np.int16)])

def run_clip(plugin, window):
    # After a wake-up the plugin ignores audio for two seconds, and reset() does not end that pause. Two seconds of
    # silence before each clip let the pause run out, so one clip's wake-up cannot hide the next clip's.
    plugin.reset()
    for _ in range(28):
        plugin.update(np.zeros(1280, np.int16).tobytes())
    plugin.found_wake_word()
    plugin.reset()
    pcm, fired, peak = to_pcm(window), False, 0.0
    for i in range(0, len(pcm), 1280):                 # 80 ms blocks
        plugin.update(pcm[i:i + 1280].tobytes())
        peak = max(peak, plugin.last_score or 0.0)
        fired = plugin.found_wake_word() or fired
    return fired, peak

plugin = WakeForgeHotwordPlugin(key_phrase=WORD_SLUG, config=dict(OVOS_HOTWORD))
print(f"Plugin loaded the head: featurizer {plugin.featurizer.name}, threshold {plugin.threshold}, patience {plugin.smoother.patience}, streaming {plugin.streaming}")
assert plugin.threshold == THRESHOLD

# ---- 1. synthetic held-out voices -------------------------------------------------------------
counts = {}
print(f"\n[1] Synthetic held-out voices\n{'kind':<12} {'detected':>12}   {'median peak score':>18}")
for kind, wins in test_audio.items():
    res = [run_clip(plugin, w) for w in wins]
    counts[kind] = (sum(f for f, _ in res), len(res))
    print(f"{kind:<12} {counts[kind][0]:>5} of {counts[kind][1]:<5} {float(np.median([p for _, p in res])):>18.3f}")
synth_hit, synth_n = counts["wake phrase"]
synth_false = sum(counts[k][0] for k in ("speech", "noise", "silence"))
synth_n_neg = sum(counts[k][1] for k in ("speech", "noise", "silence"))
print(f"Detected: {synth_hit} of {synth_n} = {synth_hit / synth_n:.1%} | false: {synth_false} of {synth_n_neg} = {synth_false / synth_n_neg:.1%}")

tick("synthetic held-out test")

In [ ]:
# ---- 2. the test stream, one unbroken feed -------------------------------------------------------------
pcm = to_int16(tst_audio)
plugin_s = WakeForgeHotwordPlugin(key_phrase=WORD_SLUG, config=dict(OVOS_HOTWORD))
fires, tst_scores, t0 = [], [], time.time()
for i in range(0, len(pcm) - HOP + 1, HOP):            # 80 ms blocks, no reset between clips
    plugin_s.update(pcm[i:i + HOP].tobytes())
    tst_scores.append(plugin_s.last_score or 0.0)
    if plugin_s.found_wake_word():
        fires.append((i + HOP) / SR)
    if (i // HOP) % 9000 == 0 and i:
        print(f"  streamed {i / SR / 60:.0f} min, {len(fires)} false wake-ups so far ({time.time() - t0:.0f} s)")
FA_PER_HOUR = len(fires) / STREAM_HOURS
FA_LO, FA_HI = poisson_ci(len(fires), STREAM_HOURS)
print(f"[2] Test stream: {len(fires)} false wake-ups in {STREAM_HOURS:.2f} h = {FA_PER_HOUR:.2f} per hour (95 % interval {FA_LO:.2f} to {FA_HI:.2f}) at threshold {THRESHOLD} (streamed in {time.time() - t0:.0f} s)")
print(f"    replaying the recorded scores with the same rule gives {len(simulate(np.array(tst_scores), THRESHOLD))} wake-ups")
by_source(fires, tst_index, "Test stream, by source")
tick("test stream")

## Cell 13 — The measurements

The cell prints what this run measured, all at the threshold chosen in Cell 11. It does not judge the numbers. `SMOKE` trains on a handful of robotic voices, so there it only checks that the head separates its own training windows (a margin of 0.1 between the medians), which an untrained head does not; that check is the test of the notebook, not a claim about the model.

In [ ]:
print("=" * 64)
print(f"Wake word {WAKE_WORD!r}   threshold {THRESHOLD}, chosen on a held-out stream ({CAL_HOURS:.2f} h): {CAL_FA:.2f} false wake-ups per hour on it")
print(f"Voices      : {len(voices['train'])} distinct in training, {len(voices['test'])} distinct in the test; positives generated in {GEN_SECONDS:.0f} s")
print(f"Synthetic   : {synth_hit} of {synth_n} held-out voices detected ({synth_hit / synth_n:.1%}); {synth_false} of {synth_n_neg} held-out speech/noise/silence clips fired")
print(f"Test stream : {len(fires)} false wake-ups in {STREAM_HOURS:.2f} h = {FA_PER_HOUR:.2f} per hour, 95 % interval {FA_LO:.2f} to {FA_HI:.2f}")
print("Run time    : " + ", ".join(f"{k} {v / 60:.0f} min" for k, v in _STEPS.items()) + f" (total {sum(_STEPS.values()) / 60:.0f} min)")
print("These numbers describe this run only; no claim is made about performance in a real home.")
print("=" * 64)

if SMOKE:
    margin = float(np.median(score_windows(head, train_audio[train_labels == 1])) - np.median(score_windows(head, train_audio[train_labels == 0])))
    print(f"SMOKE margin between training wake-phrase and other windows: {margin:.3f} (must exceed 0.1)")
    assert margin > 0.1, (f"median wake-phrase score exceeds the median other score by only {margin:.3f}; "
                          "the head did not learn to separate the phrase")

## Cell 14 — Use the model in OVOS

Copy `best_f1.onnx` to your device and add the entry below (also saved as `ovos_config.json`) to `~/.config/mycroft/mycroft.conf`. The featurizer is not copied: the plugin reads its name from the head and downloads it from the Hugging Face Hub. Install the plugin with `pip install --pre ovos-ww-plugin-wakeforge`.

**On Kaggle** the files are under the Output tab (`wakehubert_output/best_f1.onnx` and `wakehubert_output/ovos_config.json`). **On Colab** it is in the session's file panel and disappears when the session ends, so download it first.

In [ ]:
print(json.dumps(cfg, indent=2))

## What this notebook does and does not claim

- **The threshold is chosen on a held-out stream, not calibrated.** It is the lowest of a fixed list that gives at most 2 false wake-ups per hour on a calibration stream of two hours (the length is printed). That is a stated, reproducible choice. It is not the stream calibration of the published models, and a rate measured on one hour is itself uncertain.
- **The false-wake-up rate is one measurement.** It is the count over one test stream of two hours (60 % LibriSpeech sentences and Common Voice words, 40 % sound and music), in one run. Its 95 % interval is wide when the count is small. It says nothing about television, a kitchen or a living room, and an hour is too short to measure a rate below about one per hour. The published models are calibrated on much longer streams.
- **The numbers describe one run.** The detection count is on synthetic voices, and the false-wake-up rate is on one stream of public speech, sound and music. Neither says how the model performs for real speakers in a real home, and the notebook makes no such claim.
- **Voices are distinct as far as the notebook can tell.** Designed voices are fresh samples from a description; they are distinct clips, and the split cannot exclude that two descriptions sound alike. Cloned voices are identified by the LibriSpeech speaker and split by speaker.
- **Stream calibration needs a newer wakeforge.** The published models in [`OpenVoiceOS/wakehubert-wakewords`](https://huggingface.co/OpenVoiceOS/wakehubert-wakewords) have a threshold set on a long continuous stream, to a target number of false activations per hour. That calibration is in wakeforge pull request 95, which is not released. This notebook does not use `wakeforge` at all and does not reproduce those models.

In [ ]:
print(f"Threshold {THRESHOLD}: chosen on a held-out stream of {CAL_HOURS:.2f} h, not calibrated in the wakeforge sense. False wake-ups per hour were measured on one {STREAM_HOURS:.2f} h test stream only.")
print(f"Run: {'SMOKE' if SMOKE else 'full'}, device {DEVICE}, {EPOCHS} epochs, training {TRAIN_SECONDS:.0f} s.")

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.